# Pipeline de Dados - Bronze → Silver
### Transformação de dados no Data Lake

## Leitura da Bronze

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import uuid
import traceback

In [0]:
# Leitura da tabela bronze
df_bronze = spark.read.table("acidentes.bronze.acidentes_2025")

display(df_bronze)

In [0]:
df_bronze.printSchema()

## Análise exploratória

In [0]:
print("Linhas:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))

display(df_bronze.summary())

## Valores duplicados


In [0]:
# Colunas de metadados de ingestão (não considerar para duplicados)
cols_metadados = ["ingestion_timestamp", "ingestion_date", "ingestion_id", "source_table"]

# Colunas de negócio (considerar para duplicados)
cols_negocio = [c for c in df_bronze.columns if c not in cols_metadados]

total = df_bronze.count()
sem_duplicados = df_bronze.dropDuplicates(subset=cols_negocio).count()

print("Total de registros:", total)
print("Registros únicos:", sem_duplicados)
print("Duplicados:", total - sem_duplicados)

## Valores nulos ou inválidos

In [0]:
# Nulos
display(
    df_bronze.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_bronze.columns
    ])
)

In [0]:
valores_invalidos = ["", " ", "NA", "N/A", "NULL", "null", "NaN"]

# Conta valores inválidos por coluna em uma única passada (1 scan em vez de N)
invalid_exprs = [
    F.sum(
        F.when(
            (F.col(c).isNull()) | (F.trim(F.col(c)).isin(valores_invalidos)), 1
        ).otherwise(0)
    ).alias(c)
    for c in df_bronze.columns
]

invalid_row = df_bronze.select(invalid_exprs).first()

# Exibe apenas colunas com valores inválidos (>0)
for c in df_bronze.columns:
    count = invalid_row[c]
    if count > 0:
        print(f"{c}: {count}")

In [0]:
display(df_bronze.select("classificacao_acidente").distinct())

In [0]:
# Analise de colunas com valores inválidos
cols = ["classificacao_acidente", "regional", "delegacia", "uop"]

print(" Linhas com valores inválidos em classificacao_acidente")

display(
    df_bronze.filter(
        F.col("classificacao_acidente").isNull() |
        F.trim(F.col("classificacao_acidente").cast("string")).isin(valores_invalidos)
    )
)

In [0]:
# Corrigir valores inválidos de classificacao_acidente baseado em mortos/feridos
df_bronze = df_bronze.withColumn(
    "classificacao_acidente",
    F.when(
        (F.col("classificacao_acidente").isNull()) |
        (F.trim(F.col("classificacao_acidente")).isin(valores_invalidos)),
        F.when(F.col("mortos") > 0, "Com Vítimas Fatais")
         .when(F.col("feridos") > 0, "Com Vítimas Feridas")
         .otherwise("Sem Vítimas")
    ).otherwise(F.col("classificacao_acidente"))
)

print("✅ Valores de classificacao_acidente corrigidos")
display(df_bronze.select("classificacao_acidente").distinct())

In [0]:
# Não fazem sentido para a analise as colunas regional, delegacia, uop, latitude, longitude e metadados de governança
colunas_remover = ["regional", "delegacia", "uop", "latitude", "longitude", "ingestion_timestamp", "ingestion_date", "ingestion_id", "source_table"]
df = df_bronze.select([c for c in df_bronze.columns if c not in colunas_remover])
display(df)

## Analise individual de cada coluna

In [0]:
df.columns

In [0]:
# Análise de cada coluna
for col in df.columns:
    print(f"\n{'='*60}")
    print(f"COLUNA: {col}")
    print(f"Tipo: {df.schema[col].dataType}")
    
    distinct_count = df.select(col).distinct().count()
    print(f"Valores distintos: {distinct_count}")
    
    # Se for coluna categórica (poucos valores distintos), mostrar todos
    if distinct_count <= 20:
        print("Valores únicos:")
        df.select(col).distinct().orderBy(col).show(truncate=False)
    else:
        print("Amostra de valores (10 primeiros):")
        df.select(col).distinct().orderBy(col).limit(10).show(truncate=False)

In [0]:
# Verificar formato de data_inversa e horario
print("=" * 60)
print("AMOSTRA: data_inversa e horario")
print("=" * 60)
display(df.select("data_inversa", "horario").limit(20))


In [0]:
# Converter data_inversa para date (já está em formato yyyy-MM-dd)
df = df.withColumn(
    "data",
    F.to_date(F.col("data_inversa"))
)

# Criar timestamp combinando data + horario
df = df.withColumn(
    "timestamp_acidente",
    F.to_timestamp(F.concat(F.col("data_inversa"), F.lit(" "), F.col("horario")))
)

print("✅ Timestamp criado")
display(df.select("data_inversa", "horario", "data", "timestamp_acidente").limit(10))

In [0]:
# Adicionar colunas derivadas de tempo para facilitar análises
df = df.withColumn("ano", F.year("data")) \
       .withColumn("mes", F.month("data")) \
       .withColumn("dia_do_mes", F.dayofmonth("data")) \
       .withColumn("hora", F.hour("timestamp_acidente")) \
       .withColumn("periodo_dia", 
                   F.when((F.col("hora") >= 6) & (F.col("hora") < 12), "Manhã")
                    .when((F.col("hora") >= 12) & (F.col("hora") < 18), "Tarde")
                    .when((F.col("hora") >= 18) & (F.col("hora") < 24), "Noite")
                    .otherwise("Madrugada")
       )

print("✅ Colunas temporais criadas:")
print("  • ano, mes")
print("  • dia_do_mes, hora")
print("  • periodo_dia (Manhã/Tarde/Noite/Madrugada)")

display(df.select("timestamp_acidente", "ano", "mes",  "dia_do_mes", "hora", "periodo_dia").limit(10))

In [0]:
# Remover colunas originais e temporárias
df = df.drop("data_inversa", "horario_formatado")

print("✅ Colunas removidas: data_inversa")
print(f"\nTotal de colunas agora: {len(df.columns)}")
print("\nColunas temporais mantidas:")
for col in ["data", "timestamp_acidente", "ano", "mes", "dia_do_mes", "periodo_dia"]:
    print(f"  • {col}")

df.display()

In [0]:
print("=" * 60)
print("ANÁLISE: Coluna km (quilometragem)")
print("=" * 60)

# Verificar padrão de km
print("\nAmostra de valores de km:")
display(df.select("km").limit(20))

# Verificar tipos de formato
print("\nDistribuição de formatos:")
display(
    df.groupBy(F.when(F.col("km").contains(","), "Com vírgula")
                .otherwise("Sem vírgula")
               .alias("formato"))
      .count()
      .orderBy("formato")
)

print("\n" + "=" * 60)
print("ANÁLISE: Coluna br (rodovia)")
print("=" * 60)

# Verificar distribuição de br
print("\nValores mais comuns de br:")
display(df.groupBy("br").count().orderBy(F.desc("count")).limit(15))



In [0]:
# Converter km de string com vírgula para double
df = df.withColumn(
    "km_double",
    F.regexp_replace(F.col("km"), ",", ".").cast("double")
)

# Verificar a conversão
print("✅ Conversão de km para DoubleType")
print("\nComparação antes/depois:")
display(df.select("km", "km_double").limit(20))

# Verificar se há valores que falharam na conversão (NULL)
nulls_apos_conversao = df.filter(F.col("km_double").isNull()).count()
print(f"\nValores que falharam na conversão: {nulls_apos_conversao}")

# Remover coluna original
df = df.drop("km").withColumnRenamed("km_double", "km")

print("\n✅ Coluna km transformada para DoubleType")
print(f"Tipo atual: {df.schema['km'].dataType}")

In [0]:
# Verificar se br=0 é realmente uma rodovia ou dado ausente
total_registros = df.count()
total_br_zero = df.filter(F.col('br') == 0).count()

print("\nAnálise de registros com br=0:")
print(f"Total com br=0: {total_br_zero}")
print(f"Percentual: {100 * total_br_zero / total_registros:.2f}%")

# Investigar contexto de br=0
print("Amostra de registros com br=0:")
display(df.filter(F.col("br") == 0).select("br", "uf", "municipio", "km"))

# Comparar com outros valores de br
print("\nAmostra de registros com br válido:")
display(df.filter(F.col("br") > 0).select("br", "uf", "municipio", "km").limit(10))

In [0]:
# DECISÃO: Focar análise apenas em RODOVIAS FEDERAIS
# Remover registros com br=0 (acidentes fora de rodovias federais)

total_antes = df.count()
registros_br_zero = df.filter(F.col("br") == 0).count()

# Filtrar apenas rodovias federais (br > 0)
df = df.filter(F.col("br") > 0)

total_depois = df.count()

print("🎯 FOCO: Apenas acidentes em RODOVIAS FEDERAIS")
print("=" * 60)
print(f"Total de registros antes: {total_antes:,}")
print(f"Registros removidos (br=0): {registros_br_zero:,} ({100*registros_br_zero/total_antes:.2f}%)")
print(f"Total de registros depois: {total_depois:,}")

print("\nDistribuição das principais BRs:")
display(df.groupBy("br").count().orderBy(F.desc("count")).limit(15))

In [0]:
df.display()

## USO_SOLO

In [0]:
# Renomear a coluna uso_solo e substituir valores
df = df.withColumn(
    "tipo_uso_solo",  # Novo nome da coluna
    F.when(F.col("uso_solo") == "Sim", "Urbano")
     .when(F.col("uso_solo") == "Não", "Rural")
     .otherwise("Desconhecido")  # Para lidar com valores inesperados
)

# Remover a coluna original uso_solo
df = df.drop("uso_solo")

# Exibir a nova coluna
display(df.select("tipo_uso_solo").distinct())

In [0]:
# COlunas ainda não exploradas
cols_explorar = [
    "causa_acidente",
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "periodo_dia",
    "sentido_via",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via"
]

for col in cols_explorar:
    print(f"\nValores distintos de {col}:")
    display(df.select(col).distinct())

In [0]:
# Remoção de fase_dia por semelhança com periodo_dia e sentido_via por julgar desnecessário
df = df.drop("fase_dia", "sentido_via")

In [0]:
df.display()

In [0]:
#Como a coluna feridos é a soma de leves + graves vamos manter apenas ela
df = df.drop("feridos_leves", "feridos_graves")
df.display()



In [0]:
# Normalizar colunas categóricas de texto: uppercase e trim
colunas_texto = [
    "causa_acidente", "tipo_acidente", "classificacao_acidente",
    "condicao_metereologica", "tipo_pista", "tracado_via", "municipio", "uf"
]

for col in colunas_texto:
    df = df.withColumn(col, F.trim(F.upper(F.col(col))))

print("✅ Colunas de texto normalizadas (uppercase, trimmed)")
print("\nColunas normalizadas:")
for col in colunas_texto:
    print(f"  • {col}")

print("\nAmostra:")
display(df.select("municipio", "uf", "causa_acidente", "tipo_acidente").limit(10))

## VISUALIZAÇÃO

In [0]:
# Distribuição por classificação de acidente
print("📊 DISTRIBUIÇÃO DE ACIDENTES POR GRAVIDADE")
print("=" * 60)

gravidade_dist = df.groupBy("classificacao_acidente") \
                   .count() \
                   .orderBy(F.desc("count"))

display(gravidade_dist)

# Calcular percentuais
total = df.count()
print("\nPercentuais:")
for row in gravidade_dist.collect():
    print(f"  • {row['classificacao_acidente']}: {100 * row['count'] / total:.2f}%")

# Visualização gráfica
gravidade_pd = gravidade_dist.toPandas()
plt.figure(figsize=(8, 5))
plt.bar(gravidade_pd["classificacao_acidente"], gravidade_pd["count"], color="skyblue")
plt.xlabel("Classificação do Acidente")
plt.ylabel("Total de Acidentes")
plt.title("Distribuição de Acidentes por Gravidade")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Gráfico de pizza
plt.figure(figsize=(6, 6))
plt.pie(gravidade_pd["count"], labels=gravidade_pd["classificacao_acidente"], autopct='%1.1f%%', startangle=90, colors=plt.cm.Pastel1.colors)
plt.title("Percentual de Acidentes por Gravidade")
plt.tight_layout()
plt.show()

In [0]:
# Top 10 rodovias com mais acidentes
print("📊 TOP 10 RODOVIAS FEDERAIS COM MAIS ACIDENTES")
print("=" * 60)

top_brs = df.groupBy("br") \
            .agg(
                F.count("*").alias("total_acidentes"),
                F.sum("mortos").alias("total_mortos"),
                F.sum("feridos").alias("total_feridos")
            ) \
            .orderBy(F.desc("total_acidentes")) \
            .limit(10)

display(top_brs)

# Visualização gráfica
import matplotlib.pyplot as plt

top_brs_pd = top_brs.toPandas()
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(top_brs_pd["br"].astype(str), top_brs_pd["total_acidentes"], color='#ee5a6f')
ax.set_xlabel("Rodovia (BR)", fontsize=12)
ax.set_ylabel("Total de Acidentes", fontsize=12)
ax.set_title("Top 10 Rodovias Federais com Mais Acidentes", fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Databricks visualization. Run in Databricks to view.

In [0]:
# Distribuição por período do dia
print("📊 DISTRIBUIÇÃO DE ACIDENTES POR PERÍODO DO DIA")
print("=" * 60)

periodo_dist = df.groupBy("periodo_dia") \
                 .agg(
                     F.count("*").alias("total_acidentes"),
                     F.sum("mortos").alias("total_mortos"),
                     F.sum("feridos").alias("total_feridos")
                 ) \
                 .orderBy(F.desc("total_acidentes"))

display(periodo_dist)

# Visualização gráfica
import matplotlib.pyplot as plt

periodo_pd = periodo_dist.toPandas()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colors = ['#feca57', '#ff6348', '#1e3799', '#0c2461']
ax1.bar(periodo_pd["periodo_dia"], periodo_pd["total_acidentes"], color=colors)
ax1.set_xlabel("Período do Dia", fontsize=11)
ax1.set_ylabel("Total de Acidentes", fontsize=11)
ax1.set_title("Acidentes por Período", fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Gráfico de pizza
ax2.pie(periodo_pd["total_acidentes"], labels=periodo_pd["periodo_dia"], autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title("Distribuição Percentual", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

Databricks visualization. Run in Databricks to view.

In [0]:
# Distribuição por condição meteorológica
print("📊 ACIDENTES POR CONDIÇÃO METEOROLÓGICA")
print("=" * 60)

condicao_meteo = df.groupBy("condicao_metereologica") \
                   .agg(
                       F.count("*").alias("total_acidentes"),
                       F.sum("mortos").alias("total_mortos"),
                       F.sum("feridos").alias("total_feridos")
                   ) \
                   .orderBy(F.desc("total_acidentes"))

display(condicao_meteo)

# Visualização gráfica
import matplotlib.pyplot as plt

condicao_pd = condicao_meteo.toPandas()
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(condicao_pd["condicao_metereologica"], condicao_pd["total_acidentes"], color='#5f27cd')
ax.set_ylabel("Condição Meteorológica", fontsize=12)
ax.set_xlabel("Total de Acidentes", fontsize=12)
ax.set_title("Acidentes por Condição Meteorológica", fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

Databricks visualization. Run in Databricks to view.

In [0]:
# Comparação urbano vs rural
print("📊 ACIDENTES: URBANO vs RURAL")
print("=" * 60)

uso_solo_dist = df.groupBy("tipo_uso_solo") \
                  .agg(
                      F.count("*").alias("total_acidentes"),
                      F.sum("mortos").alias("total_mortos"),
                      F.sum("feridos").alias("total_feridos"),
                      F.avg("veiculos").alias("media_veiculos")
                  ) \
                  .orderBy(F.desc("total_acidentes"))

display(uso_solo_dist)

# Visualização gráfica
import matplotlib.pyplot as plt

uso_solo_pd = uso_solo_dist.toPandas()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de pizza - acidentes
colors = ['#e17055', '#74b9ff']
ax1.pie(uso_solo_pd["total_acidentes"], labels=uso_solo_pd["tipo_uso_solo"], autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title("Distribuição de Acidentes", fontsize=12, fontweight='bold')

# Gráfico de barras - mortos
ax2.bar(uso_solo_pd["tipo_uso_solo"], uso_solo_pd["total_mortos"], color=colors)
ax2.set_xlabel("Tipo de Uso do Solo", fontsize=11)
ax2.set_ylabel("Total de Mortos", fontsize=11)
ax2.set_title("Fatalidades por Tipo", fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

Databricks visualization. Run in Databricks to view.

In [0]:
# Top 10 causas de acidentes
print("📊 TOP 10 CAUSAS DE ACIDENTES")
print("=" * 60)

top_causas = df.groupBy("causa_acidente") \
               .agg(
                   F.count("*").alias("total_acidentes"),
                   F.sum("mortos").alias("total_mortos"),
                   F.sum("feridos").alias("total_feridos")
               ) \
               .orderBy(F.desc("total_acidentes")) \
               .limit(10)

display(top_causas)

# Visualização gráfica
import matplotlib.pyplot as plt

top_causas_pd = top_causas.toPandas()
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top_causas_pd["causa_acidente"], top_causas_pd["total_acidentes"], color='#f39c12')
ax.set_ylabel("Causa do Acidente", fontsize=11)
ax.set_xlabel("Total de Acidentes", fontsize=11)
ax.set_title("Top 10 Causas de Acidentes", fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

Databricks visualization. Run in Databricks to view.

In [0]:
# Evolução temporal mensal
print("📊 EVOLUÇÃO TEMPORAL DE ACIDENTES (2025)")
print("=" * 60)

evolucao_mensal = df.groupBy("mes") \
                    .agg(
                        F.count("*").alias("total_acidentes"),
                        F.sum("mortos").alias("total_mortos"),
                        F.sum("feridos").alias("total_feridos")
                    ) \
                    .orderBy("mes")

display(evolucao_mensal)

# Visualização gráfica
import matplotlib.pyplot as plt

evolucao_pd = evolucao_mensal.toPandas()
meses_labels = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(evolucao_pd["mes"], evolucao_pd["total_acidentes"], marker='o', linewidth=2, markersize=8, color='#e74c3c')
ax.fill_between(evolucao_pd["mes"], evolucao_pd["total_acidentes"], alpha=0.3, color='#e74c3c')
ax.set_xlabel("Mês", fontsize=12)
ax.set_ylabel("Total de Acidentes", fontsize=12)
ax.set_title("Evolução Temporal de Acidentes em 2025", fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_labels)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Databricks visualization. Run in Databricks to view.

In [0]:
# Top 10 tipos de acidentes
print("📊 TOP 10 TIPOS DE ACIDENTES")
print("=" * 60)

top_tipos = df.groupBy("tipo_acidente") \
              .agg(
                  F.count("*").alias("total_acidentes"),
                  F.sum("mortos").alias("total_mortos"),
                  F.sum("feridos").alias("total_feridos")
              ) \
              .orderBy(F.desc("total_acidentes")) \
              .limit(10)

display(top_tipos)

# Visualização gráfica
import matplotlib.pyplot as plt

top_tipos_pd = top_tipos.toPandas()
fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.Spectral(range(len(top_tipos_pd)))
ax.barh(top_tipos_pd["tipo_acidente"], top_tipos_pd["total_acidentes"], color=colors)
ax.set_ylabel("Tipo de Acidente", fontsize=11)
ax.set_xlabel("Total de Acidentes", fontsize=11)
ax.set_title("Top 10 Tipos de Acidentes", fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

Databricks visualization. Run in Databricks to view.

## 📦 Armazenamento na Camada Silver com Governança

In [0]:
import uuid
import traceback

def ingest_to_silver(df_transformado, nome_tabela_origem: str, nome_tabela_destino: str, particionar_por: list = None):
    """
    Função para salvar dataframe transformado na camada Silver com governança.
    
    Args:
        df_transformado: DataFrame já transformado e limpo
        nome_tabela_origem: Nome da tabela de origem (bronze)
        nome_tabela_destino: Nome da tabela de destino (silver)
        particionar_por: Lista de colunas para particionamento (ex: ["ano", "mes"])
    """
    try:
        origem = f"acidentes.bronze.{nome_tabela_origem}"
        destino = f"acidentes.silver.{nome_tabela_destino}"

        print(f"📍 Origem: {origem}")
        print(f"🎯 Destino: {destino}")

        # Criar schema silver se não existir
        spark.sql("CREATE SCHEMA IF NOT EXISTS acidentes.silver")
        print("✅ Schema acidentes.silver verificado")

        # Dropar tabela existente para evitar conflitos de schema
        spark.sql(f"DROP TABLE IF EXISTS {destino}")
        print("✅ Tabela antiga removida (se existia)")

        # Versionamento
        ingestion_id = str(uuid.uuid4())

        # Adicionar colunas de controle
        df_silver = (
            df_transformado
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("ingestion_id", F.lit(ingestion_id))
            .withColumn("source_table", F.lit(nome_tabela_origem))
            .withColumn("processing_date", F.current_date())
        )

        # Escrita Silver com particionamento (se especificado)
        writer = df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true")
        
        if particionar_por:
            writer = writer.partitionBy(*particionar_por)
            print(f"📋 Particionamento: {', '.join(particionar_por)}")
        
        writer.saveAsTable(destino)

        # LOG DE INGESTÃO
        qtd_linhas = df_silver.count()

        log_df = spark.createDataFrame([(
            ingestion_id,
            origem,
            destino,
            qtd_linhas,
            "silver"
        )], ["ingestion_id", "source_table", "target_table", "row_count", "layer"])

        log_df = log_df.withColumn("log_timestamp", F.current_timestamp())

        log_df.write.mode("append").saveAsTable("acidentes.silver.ingestion_log")

        print(f"✅ Ingestion ID: {ingestion_id}")
        print(f"✅ Registros salvos: {qtd_linhas:,}")
        print("🎉 Silver criada com sucesso!\n")

    except Exception as e:
        print(f"⚠️ Erro na ingestão Silver: {str(e)}")
        traceback.print_exc()
        raise

print("✅ Função ingest_to_silver() definida")

In [0]:
# Executar ingestão usando a função
print("=" * 60)
print("🚀 INICIANDO INGESTÃO BRONZE → SILVER")
print("=" * 60)
print()

ingest_to_silver(
    df_transformado=df,
    nome_tabela_origem="acidentes_2025",
    nome_tabela_destino="acidentes_2025",
    particionar_por=["ano", "mes"]
)

print("🎉 Pipeline Bronze → Silver concluído com sucesso!")

In [0]:
print("🔍 VALIDAÇÃO DA CAMADA SILVER")
print("=" * 60)

# Listar tabelas
print("\nTabelas no schema silver:")
spark.sql("SHOW TABLES IN acidentes.silver").show(truncate=False)

# Verificar estrutura
print("\nSchema da tabela silver:")
df_validacao = spark.table("acidentes.silver.acidentes_2025")
df_validacao.printSchema()

# Amostra de dados
print("\nAmostra de 5 registros:")
display(df_validacao.limit(5))

# Estatísticas
print(f"\n📊 ESTATÍSTICAS:")
print(f"Total de registros: {df_validacao.count():,}")
print(f"Total de colunas: {len(df_validacao.columns)}")
print(f"Partições: ano, mes")

# Verificar distribuição por partição
print("\nDistribuição por partição (ano/mês):")
display(df_validacao.groupBy("ano", "mes").count().orderBy("ano", "mes"))

In [0]:
print("📋 LOG DE INGESTÃO SILVER")
print("=" * 60)

log_silver = spark.table("acidentes.silver.ingestion_log")
display(log_silver.orderBy(F.desc("log_timestamp")))

print("\n✅ Pipeline completo documentado e rastreado!")